## Batch scripts: Utilities and Packages

In [ ]:
# parameters
batch_number = 0

In [ ]:
# batch_number=1

In [ ]:
import os
import sys
## adding the root directory to the library directory so still can read utility functions
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("..."), "..")))
project_root = os.path.abspath(os.path.expanduser("..."))
sys.path.append(project_root)
os.chdir(project_root) # setting work directory

import matplotlib.pyplot as plt
import warnings
import pickle 
import pandas as pd
import numpy as np

from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import xgboost as xgb


def fxn():
    warnings.warn("deprecated", DeprecationWarning)

with warnings.catch_warnings():
    
    warnings.simplefilter("ignore")
    fxn()

In [ ]:
# Load Olink and Soma assays from the MESA cohort

MESA_Olink  = pd.read_csv("Data/final_datasets/MESA_Olink_3k_log2_nodup_rm10na_knn_final.tsv.gz", sep = "\t", 
                          compression = "gzip")
MESA_Soma = pd.read_csv("Data/final_datasets/MESA_SOMA_5k_log2scaled_nodup_final.tsv.gz", sep = "\t", 
                        compression = "gzip").drop(columns={"bmi"}, axis=1)

In [ ]:
# Find subjects who are in both datasets

overlap_IDs = np.unique(list(set(np.array(MESA_Olink["id"])).intersection(set(np.array(MESA_Soma["id"])))))

In [ ]:
# Drop repeated subject IDs in both cohorts

MESA_Olink  = MESA_Olink[MESA_Olink["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])
MESA_Soma   = MESA_Soma[MESA_Soma["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])


print(MESA_Olink.shape)
print(MESA_Soma.shape)

ids1 = list(MESA_Olink["id"])
ids2 = list(MESA_Soma["id"])

# fools check in checking if rows are aligned or not
if ids1 == ids2:
    print("ids in the same order")
else:
    print("ids not in the same order")

# making sure the outcomes index are the same as training index for train test split

In [ ]:
# Compare the Olink and Soma measurements of LEPTIN protin

# # makesure protein list are in the same order for soma and olink
# MESA_Soma_aligned, MESA_Olink_aligned = MESA_Soma.align(MESA_Olink, join='inner', axis=0)

MESA_Soma = MESA_Soma.set_index("id")
MESA_Olink = MESA_Olink.set_index("id")


if MESA_Soma.index.equals(MESA_Olink.index):
    print("yes") 
else:
    print("no")

    
    
print(MESA_Soma.index)
print(MESA_Olink.index)

In [ ]:
## loading in training and testing index
train_index = pickle.load(open(f"Data/helper_data/train_index.pkl", "rb"))
test_index = pickle.load(open(f"Data/helper_data/test_index.pkl", "rb"))

print(len(test_index))

In [ ]:
## loading prot list
protlist = pickle.load(open(f"Data/helper_data/protlist_missing.pkl", "rb"))
# protlist = pickle.load(open(f"Data/helper_data/protlist_chunk{batch_number}.pkl", "rb"))
print(protlist[0:10])
print(len(protlist))

## Training loop for ?cores

In [ ]:
## defining X
X=MESA_Soma
print(MESA_Soma.shape)

In [ ]:
# ### stress test, how long does it take for 10 proteins to train?
# protlist = protlist[:10]
# protlist

In [ ]:
## define function to train for GRADIENT BOOST
def train_soma_to_olink(protein_name):
    # extract the outcome
    y = MESA_Olink[protein_name]
    # if there are any NA in y skip first
    if y.loc[train_index].isna().any():
        print(f"skip protein {protein_name} due to NA in olink data")
        return
    
    # use the X from somascan to predict the olink outcome
    X_train = X.loc[train_index]
    y_train = y.loc[train_index]
    
    ## gradient boost
    ## this mimics the default for sklearn's setting
    reg = HistGradientBoostingRegressor(random_state=0)
    reg.fit(X_train, y_train)

    # Save model to file (e.g., saved_models/IL6.pkl)
    model_path = f"Data/soma_trained_models/soma_olink/5k_log2scaled_soma_3kolink/{protein_name}_hgb.pkl"
    pickle.dump(reg, open(model_path, "wb"))
    print(f"Saved model for {protein_name} to {model_path}")

In [ ]:
%%time

for p in protlist:
    train_soma_to_olink(p)